In [1]:
# !pip install tensorboardx -q

In [2]:
import gymnasium as gym
from collections import defaultdict, Counter
from tensorboardX import SummaryWriter

ENV_NAME = "FrozenLake-v1"
GAMMA = 0.9
TEST_EPISODES = 20

In [8]:
class Agent:
  def __init__(self):
    self.env = gym.make(ENV_NAME)
    self.state, _ = self.env.reset()
    self.rewards = defaultdict(float)
    self.transits = defaultdict(Counter)
    self.values = defaultdict(float)

  def play_n_random_steps(self, count):
     for _ in range(count):
      action = self.env.action_space.sample()
      new_state, reward, terminated, truncated, _ = self.env.step(action)
      self.rewards[(self.state, action, new_state)] = reward
      self.transits[(self.state, action)][new_state] += 1
      self.state = self.env.reset()[0] if terminated else new_state

  def calc_action_value(self, state, action):
    target_counts = self.transits[(state,action)]
    total = sum(target_counts.values())
    action_value = 0.0
    for tgt_state, count in target_counts.items():
      reward = self.rewards[(state, action, tgt_state)]
      val = reward + GAMMA * self.values[tgt_state]
      action_value += (count / total) * val

    return action_value

  def select_action(self, state):
    best_action, best_value = None, None
    for action in range(self.env.action_space.n):
      action_value = self.calc_action_value(state, action)
      if best_value is None or best_value < action_value:
        best_value = action_value
        best_action = action
    return best_action

  def play_episode(self, env):
    total_reward = 0.0
    state, _ = env.reset()
    while True:
      action = self.select_action(state)
      new_state, reward, terminated, truncated, _ = env.step(action)
      self.rewards[(state, action, new_state)] = reward
      self.transits[(state, action)][new_state] += 1
      total_reward += reward
      if terminated:
        break
      state = new_state
    return total_reward

  def value_iteration(self):
    for state in range(self.env.observation_space.n):
      state_values = [
          self.calc_action_value(state, action)
          for action in range(self.env.action_space.n)
      ]
      self.values[state] = max(state_values)

In [9]:
if __name__ == "__main__":
  test_env = gym.make(ENV_NAME)
  agent = Agent()
  writer = SummaryWriter(comment="-v-iteration")
  iter_no = 0
  best_reward = 0.0
  while True:
    iter_no += 1
    agent.play_n_random_steps(100)
    agent.value_iteration()

    reward = 0.0
    for _ in range(TEST_EPISODES):
      reward += agent.play_episode(test_env)
    reward /= TEST_EPISODES
    writer.add_scalar("reward", reward, iter_no)
    if reward > best_reward:
      print(f"Best reward updated {best_reward:.3f} -> {reward:.3f}")
      best_reward = reward
    if reward > 0.80:
      print(f"Solved in {iter_no} iterations!")
      break

  writer.close()

Best reward updated 0.000 -> 0.100
Best reward updated 0.100 -> 0.500
Best reward updated 0.500 -> 0.650
Best reward updated 0.650 -> 0.700
Best reward updated 0.700 -> 0.850
Solved in 19 iterations!
